# 10 · Demo: URL detector, attack and defense (string-only)

**T4-03 · TV4 – Nguyễn Trần Đan Huy (SE211085)**

What this demo does
1. **Score**: paste a URL → score and decision (BLOCK / ALLOW) for two configurations, `none` and `D1+D2+D3`, at the **locked** thresholds.
2. **Try attack**: greedy black-box attack, budget k = 3, against the `none` configuration → shows the edited URL, the operations used, and the new score under both configurations.

Safety rules (Phân công, Mục 1.4)
- The demo **only processes strings**. It never opens, fetches or resolves a URL. Every scoring and attack call runs inside an `offline()` guard that blocks outbound connections.
- URLs are always **displayed defanged** (`hxxp`, `[.]`). You can type either form.
- No locked file is edited. This notebook writes **nothing** to `results.csv` (no `log_result` call).

Run the cells top to bottom. Needs notebooks 01–07 to have been run first (the models and thresholds of all five configurations must exist). Cell 6 is a self-test: it must print SELF-TEST PASSED before the demo is shown.

In [ ]:
# Ô 1 – standard group cell (for a repro check change ONLY the ROOT line)
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/AIC211_URL'
import sys
sys.path.insert(0, ROOT + '/src')

In [ ]:
import os
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'   # no telemetry / version check
# same libraries as the Ô 1 of notebooks 05-08, plus gradio
%pip install -q tldextract rapidfuzz shap xgboost gradio
import gradio as gr
print('gradio', gr.__version__, '  <- copy this version into requirements.txt')

In [ ]:
# Helpers that do NOT depend on the shared modules: offline guard + defang/refang
import re, socket, contextlib

class NetworkBlocked(ConnectionError):      # a ConnectionError, so libraries treat it as "offline" and fall back
    pass

_LOCAL = {'localhost', '127.0.0.1', '::1', ''}
blocked_attempts = {'n': 0}

def _local_host(h):
    return h is None or str(h) in _LOCAL or str(h).startswith('127.')

def _local_addr(addr):
    host = addr[0] if isinstance(addr, (tuple, list)) and addr else None   # AF_UNIX paths -> None
    return _local_host(host)

@contextlib.contextmanager
def offline():
    """Inside this block any non-loopback connection or DNS lookup raises NetworkBlocked."""
    real_connect, real_connect_ex, real_gai = socket.socket.connect, socket.socket.connect_ex, socket.getaddrinfo
    def _block():
        blocked_attempts['n'] += 1
        raise NetworkBlocked('outbound network access is blocked in this demo')
    def connect(self, addr, *a, **kw):
        if not _local_addr(addr): _block()
        return real_connect(self, addr, *a, **kw)
    def connect_ex(self, addr, *a, **kw):
        if not _local_addr(addr): _block()
        return real_connect_ex(self, addr, *a, **kw)
    def gai(host, *a, **kw):
        if not _local_host(host): _block()
        return real_gai(host, *a, **kw)
    socket.socket.connect, socket.socket.connect_ex, socket.getaddrinfo = connect, connect_ex, gai
    try:
        yield
    finally:
        socket.socket.connect, socket.socket.connect_ex, socket.getaddrinfo = real_connect, real_connect_ex, real_gai

def defang_text(s):
    """Idempotent safety net: hxxp scheme everywhere, [.] in the host part."""
    s = re.sub(r'(?i)\bhttp(s?)(?=://)', lambda m: 'hxxp' + m.group(1), s)
    pre, host, rest = re.match(r'^((?:hxxps?://)?)([^/?#]*)(.*)$', s, flags=re.S).groups()
    host = re.sub(r'(?<!\[)\.(?!\])', '[.]', host)
    return pre + host + rest

def refang(s):
    s = s.strip()
    s = re.sub(r'(?i)^hxxp(s?)(?=://)', lambda m: 'http' + m.group(1), s)
    return s.replace('[.]', '.')

assert defang_text('http://a.b.com/x.y?r=https://c.d') == 'hxxp://a[.]b[.]com/x.y?r=hxxps://c.d'
assert defang_text(defang_text('https://a.b.com/')) == 'hxxps://a[.]b[.]com/'
assert refang('hxxps://a[.]b[.]com/x') == 'https://a.b.com/x'
try:
    with offline():
        socket.getaddrinfo('example.invalid', 80)
    raise SystemExit('offline() guard did not block DNS')
except NetworkBlocked:
    print('helpers OK; offline guard blocks outbound lookups')

In [ ]:
# Load shared code and data exactly as notebooks 05-08 do (read only; nothing is written)
import json, random, importlib
import numpy as np, pandas as pd
import common, attacks, e3, e4
for m in (common, attacks, e3, e4):
    importlib.reload(m)

def _pick(name):
    for m in (attacks, e3, common):
        if hasattr(m, name):
            return getattr(m, name)
    raise ImportError(f'{name} not found in attacks / e3 / common')

normalize_url, to_attack_form, defang = _pick('normalize_url'), _pick('to_attack_form'), _pick('defang')
attack_success_rate = _pick('attack_success_rate')
ALL_OPS = list(_pick('FREE_OPS')) + list(_pick('PAID_OPS'))       # same as notebooks 05-08

A = pd.read_parquet(f'{ROOT}/data/processed/A.parquet')
T = A[A['split'] == 'test']
benign_domains = set(A[A.label == 0]['reg_domain'])

CFG = e4.load_configs(ROOT)                  # {name: {'thr': float, 'score': fn(list of 'http://…') -> scores}}
missing = [n for n in ('none', 'D1+D2+D3') if n not in CFG]
assert not missing, f'Missing configuration(s) {missing}: run notebooks 03, 06 and 07 first.'
print('configurations:', {n: round(c['thr'], 4) for n, c in CFG.items()})

In [ ]:
# Demo functions. Conventions copied from the notebooks:
#   score input  = 'http://' + url_norm        (e4.clean_eval)
#   decision     = BLOCK if score > thr        (notebook 08, cell 5)
#   attack input = to_attack_form(url_norm)    (notebooks 05-08, the 300 attacked URLs)
SHOWN = ['none', 'D1+D2+D3']
MAX_LEN = 2048                                  # same limit as attacks.is_valid

class InputError(ValueError):
    """Message written by this demo (never contains a URL), safe to show as is."""

def to_norm(text):
    """Same normalisation as the pipeline (no input limits): used by the self-test on dataset rows."""
    return normalize_url(refang(text))          # string processing only

def prepare(text):
    """to_norm + input checks for text typed into the demo."""
    if not isinstance(text, str) or not text.strip():
        raise InputError('Enter a URL.')
    raw = refang(text)
    if len(raw) > MAX_LEN:
        raise InputError(f'URL longer than {MAX_LEN} characters.')
    if re.search(r'\s', raw):
        raise InputError('A URL cannot contain whitespace.')
    return to_norm(raw)

def show(u):
    try:
        u = defang(u)
    except Exception:
        pass
    return defang_text(str(u))

def score(name, url_norms):
    return np.asarray(CFG[name]['score'](['http://' + u for u in url_norms]), dtype=float)

def decision(s, thr):
    return 'BLOCK' if s > thr else 'ALLOW'

def greedy_attack(url_norm, k=3):
    """Greedy black-box attack on 'none' (same call as notebooks 05-08, on one URL)."""
    cfg = CFG['none']
    _, res = attack_success_rate([to_attack_form(url_norm)], cfg['score'], cfg['thr'], k,
                                 ALL_OPS, 'greedy', benign_domains, seed=0)
    if len(res) == 0:
        raise InputError('The attack returned no result for this URL.')
    r = res.iloc[0]
    return {'adv': str(r.adv_url), 'ops': list(r.ops), 'queries': int(r.queries), 'evaded': bool(r.evaded)}

In [ ]:
# Self-test: all checks must pass before the demo is shown.
# 1. locked thresholds (team documents: none = RF_F1 0.7613, D1+D2+D3 0.7839)
assert abs(CFG['none']['thr'] - 0.7613) < 5e-4, CFG['none']['thr']
assert abs(CFG['D1+D2+D3']['thr'] - 0.7839) < 5e-4, CFG['D1+D2+D3']['thr']

# 2. to_norm() reproduces the pipeline's own url_norm (dataset rows may contain spaces, so prepare() is not used here)
sample = T.sample(n=min(1000, len(T)), random_state=0)
match = float(np.mean([to_norm(u) == n for u, n in zip(sample['url'], sample['url_norm'])]))
print(f'to_norm() == url_norm on {len(sample)} test rows: {match:.3f}')
assert match >= 0.999, 'to_norm() does not reproduce url_norm'

# 3. sanity: FPR on benign A-test URLs should be about 1% (documents: 0.8% / 0.9% on the full A-test)
ben = T[T['label'] == 0].sample(n=min(3000, int((T['label'] == 0).sum())), random_state=0)
blocked_attempts['n'] = 0
with offline():
    for name in SHOWN:
        fpr = float(np.mean(score(name, ben['url_norm']) > CFG[name]['thr']))
        print(f'FPR on {len(ben)} benign A-test URLs, {name}: {fpr:.3%}')
        if not 0.0 <= fpr <= 0.03:
            print('  WARNING: far from 1% -> check the score input form or the > convention')

# 4. offline: scoring made no outbound connection
print('blocked outbound attempts during scoring:', blocked_attempts['n'],
      '(0 expected; if > 0 a library tried to go online and was blocked - tell TV4)')
print('SELF-TEST PASSED')

In [ ]:
# Three sample URLs from A-test (seeded), shown defanged. Chosen so the attack button is interesting.
rng = random.Random(0)
pool_b = T[T['label'] == 0]['url_norm'].tolist()
pool_m = T[T['label'] == 1]['url_norm'].tolist()
rng.shuffle(pool_b); rng.shuffle(pool_m)
pool_b, pool_m = pool_b[:300], pool_m[:300]
thr_none = CFG['none']['thr']
with offline():
    s_b, s_m = score('none', pool_b), score('none', pool_m)
benign_ok = [u for u, s in zip(pool_b, s_b) if s <= thr_none][:1]          # clean URL, allowed
caught    = [u for u, s in zip(pool_m, s_m) if s > thr_none][:2]           # malicious, caught by 'none'
assert len(benign_ok) == 1 and len(caught) == 2, 'could not pick sample URLs'
SAMPLES = [('benign (A-test)', benign_ok[0]), ('malicious, caught (A-test)', caught[0]), ('malicious, caught (A-test)', caught[1])]
for label, u in SAMPLES:
    print(f'{label:28s}', show(u))

In [ ]:
import gradio as gr

HEAD = ['Configuration', 'Score', 'Threshold', 'Decision']
ATK_HEAD = ['Configuration', 'Score before', 'Decision before', 'Score after', 'Decision after']

def _err(e):
    if isinstance(e, InputError):
        return f'**Error:** {e}', pd.DataFrame(columns=HEAD)
    kind = 'Error' if isinstance(e, NetworkBlocked) else f'Unexpected {type(e).__name__}'
    return f'**{kind}:** {defang_text(str(e))}', pd.DataFrame(columns=HEAD)   # library messages may contain a URL

def analyze(text):
    try:
        u = prepare(text)
        with offline():
            rows = []
            for name in SHOWN:
                s = float(score(name, [u])[0]); thr = CFG[name]['thr']
                rows.append([name, f'{s:.4f}', f'{thr:.4f}', decision(s, thr)])
        return f'Normalised (defanged, never opened): `{show(u)}`', pd.DataFrame(rows, columns=HEAD)
    except Exception as e:
        return _err(e)

def try_attack(text):
    try:
        u = prepare(text)
        with offline():
            before = {n: float(score(n, [u])[0]) for n in SHOWN}
            if before['none'] <= CFG['none']['thr']:
                return 'The URL is already ALLOWED by `none`; the attack only edits URLs that are blocked.', pd.DataFrame(columns=ATK_HEAD)
            res = greedy_attack(u, k=3)
            adv = prepare(res['adv'])
            after = {n: float(score(n, [adv])[0]) for n in SHOWN}
        rows = [[n, f'{before[n]:.4f}', decision(before[n], CFG[n]['thr']),
                 f'{after[n]:.4f}', decision(after[n], CFG[n]['thr'])] for n in SHOWN]
        ops = ', '.join(defang_text(str(o)) for o in res['ops']) or '(none)'
        verdict = 'evaded `none`' if res['evaded'] else 'did not evade `none` within k = 3'
        msg = (f'Original: `{show(u)}`  \n'
               f'Modified: `{show(adv)}`  \n'
               f'Operations used: {ops}  \n'
               f'Queries to the detector: {res["queries"]}  \n'
               f'Result: {verdict}. Greedy black box, k = 3, run against `none` only; both configurations score the modified URL.')
        return msg, pd.DataFrame(rows, columns=ATK_HEAD)
    except Exception as e:
        return _err(e)

with gr.Blocks(title='AIC211 Topic 2 – URL detector demo', analytics_enabled=False) as demo:
    gr.Markdown('## URL detector demo (string-only)\n'
                'Nothing is opened, fetched or resolved. URLs are shown defanged. You may type `hxxp://…` and `[.]`.')
    url_in = gr.Textbox(label='URL (defanged or plain)', placeholder='hxxps://example[.]com/path?id=1')
    gr.Examples(examples=[[show(u)] for _, u in SAMPLES], inputs=url_in, label='Sample URLs from A-test (defanged)')
    with gr.Row():
        b1 = gr.Button('Score', variant='primary')
        b2 = gr.Button('Try attack (greedy, k = 3)')
    out_md = gr.Markdown()
    out_tbl = gr.Dataframe(interactive=False, wrap=True)
    b1.click(analyze, url_in, [out_md, out_tbl])
    b2.click(try_attack, url_in, [out_md, out_tbl])

demo.launch(share=False, debug=False)     # loopback only, no public link

**If Colab shows only a local address** (no interface below the previous cell), run this once in a new cell:

```python
from google.colab import output
output.serve_kernel_port_as_iframe(demo.server_port, height=800)
```

**For the report / video:** the three samples come from A-test with `random.Random(0)`, so the demo is repeatable. Record the video with the sample buttons, not with real URLs from the wild.